# RecomendaAI - Treinamento do Modelo de Recomendação

Este notebook demonstra o processo de treinamento e avaliação dos modelos de recomendação para o projeto RecomendaAI.

Abordagens:
1. **Content-Based Filtering**: Baseado nos metadados dos filmes (gêneros e sinopse).
2. **Collaborative Filtering**: Baseado no comportamento do usuário (avaliações).

In [1]:
import pandas as pd
import numpy as np
import json
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from surprise import SVD, Dataset, Reader, accuracy
from surprise.model_selection import train_test_split
import pickle

print("✅ Bibliotecas carregadas!")

✅ Bibliotecas carregadas!


## 1. Carregamento de Dados

In [2]:
def load_movies(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        return json.load(f)

movies_data = load_movies('../data/tmdb_movies_large.json')
df_movies = pd.DataFrame(movies_data)
print(f"🎬 {len(df_movies)} filmes carregados.")
df_movies.head(3)

🎬 696 filmes carregados.


,id,title,original_title,overview,release_date,vote_average,vote_count,genres,poster_path,backdrop_path
0,1007734,Anônimo 2,Nobody 2,"O pai suburbano Hutch Mansell, ex-assassino le...",2025-08-13,7.239,394,"[Ação, Thriller]",/6ObRyIQHJAY2X575hLVgKsnNlar.jpg,/mEW9XMgYDO6U0MJcIRqRuSwjzN5.jpg
1,1035259,Corra Que a Polícia Vem Aí!,The Naked Gun,Apenas um homem tem as habilidades necessárias...,2025-07-30,6.774,556,"[Ação, Comédia, Crime]",/aGnR0XntfMrlrbnVyPL8XOKAkAQ.jpg,/1wi1hcbl6KYqARjdQ4qrBWZdiau.jpg
2,1311031,Demon Slayer: Kimetsu no Yaiba - Castelo Infinito,劇場版「鬼滅の刃」無限城編 第一章 猗窩座再来,"Japão, era Taisho. Tanjiro, um bondoso jovem q...",2025-07-18,7.400,106,"[Animação, Ação, Fantasia, Thriller]",/c55sXCaQBj3vuHqZe62tv90xCQS.jpg,/1RgPyOhN4DRs225BGTlHJqCudII.jpg


## 2. Content-Based Filtering (TF-IDF)
Vamos usar gêneros e sinopse para encontrar filmes similares.

In [ ]:
import nltk
from nltk.corpus import stopwords

    df_movies['genres_str'] = df_movies['genres'].apply(lambda x: ' '.join(x))
    df_movies['content'] = df_movies['genres_str'] + ' ' + df_movies['overview'].fillna('')

    try:
        portuguese_stopwords = stopwords.words('portuguese')
    except Exception:
        nltk.download('stopwords')
        portuguese_stopwords = stopwords.words('portuguese')

    tfidf = TfidfVectorizer(stop_words=portuguese_stopwords)
    tfidf_matrix = tfidf.fit_transform(df_movies['content'])

    print(f"Matrix TF-IDF gerada: {tfidf_matrix.shape}")

    cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

    def get_content_recommendations(title, n=10):
        title_lower = title.lower()
        matches = df_movies[df_movies['title'].str.lower() == title_lower]

        if matches.empty:
            matches = df_movies[df_movies['title'].str.lower().str.contains(title_lower)]
            if matches.empty:
                print(f"❌ Filme '{title}' não encontrado.")
                return []

        idx = matches.index[0]
        matched_title = matches.iloc[0]['title']
        print(f"🎬 Filme de referência: '{matched_title}'")

        sim_scores = list(enumerate(cosine_sim[idx]))
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
        sim_scores = sim_scores[1:n+1]

        movie_indices = [i[0] for i in sim_scores]
        return df_movies.iloc[movie_indices][['id', 'title', 'vote_average']]

    print("🎯 Exemplo de recomendação para 'Anônimo 2':")
    print(get_content_recommendations('Anônimo 2', n=5))

Matrix TF-IDF gerada: (696, 7723)
🎯 Exemplo de recomendação para 'Anônimo 2':
🎬 Filme de referência: 'Anônimo 2'
          id                  title  vote_average
19    615457                Anônimo         7.911
10   1151334      Motorista de Fuga         6.412
44   1029575       Plano em Família         7.262
688      301  Onde Começa o Inferno         7.812
397   646385                 Pânico         6.685


## 3. Collaborative Filtering (SVD)
Como não temos um dataset de avaliações real, vamos gerar um sintético para demonstração.

In [ ]:
np.random.seed(42)
num_users = 100
ratings_data = []
movie_ids = df_movies['id'].tolist()

for user_id in range(1, num_users + 1):
    num_user_ratings = np.random.randint(10, 50)
    chosen_movies = np.random.choice(movie_ids, num_user_ratings, replace=False)
    for movie_id in chosen_movies:
        rating = np.random.uniform(1, 5)
        ratings_data.append([user_id, movie_id, rating])

df_ratings = pd.DataFrame(ratings_data, columns=['user_id', 'item_id', 'rating'])
print(f"📊 {len(df_ratings)} avaliações sintéticas geradas.")

reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(df_ratings[['user_id', 'item_id', 'rating']], reader)

trainset, testset = train_test_split(data, test_size=0.2)
algo = SVD()
algo.fit(trainset)

predictions = algo.test(testset)
print(f"📉 RMSE: {accuracy.rmse(predictions):.4f}")

# Exemplo de predição para o usuário 1 e o filme 1035259
pred = algo.predict(1, 1035259)
print(f"🔮 Predição para Usuário 1 no filme {pred.iid}: {pred.est:.2f}")

#   • RMSE = 0: Seria o modelo perfeito. Ele acertou exatamente todas as notas de todos os usuários (raríssimo ou sinal de overfitting).
#   • RMSE menor que 1.0: Excelente! O modelo erra muito pouco (menos de 1 estrela na média).
#   • RMSE entre 1.0 e 1.2: Muito bom para sistemas de recomendação reais (é o patamar do nosso modelo).
#   • RMSE maior que 1.5: O modelo está chutando muito mal e precisa de ajustes.

📊 2907 avaliações sintéticas geradas.
RMSE: 1.1733
📉 RMSE: 1.1733
🔮 Predição para Usuário 1 no filme 1035259: 3.09


## 4. Exportação dos Modelos

In [6]:
os.makedirs('../models/weights', exist_ok=True)

# Salvar o modelo SVD
with open('../models/weights/svd_model.pkl', 'wb') as f:
    pickle.dump(algo, f)

# Salvar a matriz TF-IDF e o vetorizador para o Content-Based
with open('../models/weights/tfidf_model.pkl', 'wb') as f:
    pickle.dump((tfidf, tfidf_matrix), f)

print("💾 Modelos salvos em models/weights/")

💾 Modelos salvos em models/weights/
